Exercise 2: Flow Matching on MNIST



[← Back to reading list]()

# Exercise 2: Flow Matching on MNIST

> **Learning objectives:**

**Learning objectives:**


- Implement patchify/unpatchify to convert images into token sequences

- Build a Diffusion Transformer (DiT) with adaptive layer norm conditioning

- Understand and implement the rectified flow matching objective

- Train a model to generate MNIST digits from noise

- Add class conditioning and classifier-free guidance (CFG)

**Prerequisites:** [Exercise 1]() (basic transformer). Familiarity with the [Scaling Rectified Flow Transformers]() paper is helpful but not required. Reference implementation: [wendlerc/mnist]().

## Setup

In [ ]:
import torch
import torch as t
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from datasets import load_dataset
import matplotlib.pyplot as plt
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"

## Part A: Patchifying Images

Vision transformers don't process raw pixels. Instead, they split images into non-overlapping patches and project each patch into an embedding vector. A 28×28 MNIST image with patch size 4 becomes a sequence of \((28/4)^2 = 49\) tokens (we'll pad to 32×32 first, giving \((32/4)^2 = 64\) tokens, but using the raw 28×28 with a strided convolution that handles the alignment is cleaner—we'll use `padding=2` to make it work with patch_size=4, yielding \(7 \times 7 = 49\) tokens).

---

### Exercise 1 — Implement Patch (Image → Tokens)

Difficulty: 🔴🔴🔴⭕⭕  |  Importance: 🔵🔵🔵🔵🔵  |  ~15 min

Convert an image of shape `(batch, channels, height, width)` into a sequence of patch embeddings of shape `(batch, n_patches, d)`. Use a 2D convolution with `stride=patch_size` to extract and embed patches in one operation.

In [ ]:
class Patch(nn.Module):
    """Convert images to patch token sequences.

    A Conv2d with stride=patch_size extracts one patch per spatial position.
    The output channels of the conv become the embedding dimension.
    """
    def __init__(self, patch_size=4, in_channels=1, d=32):
        super().__init__()
        self.d = d
        self.patch_size = patch_size
        # YOUR CODE HERE
        # Create a Conv2d that maps (batch, in_channels, H, W)
        # to (batch, d, H//patch_size, W//patch_size)
        # Use kernel_size=5, padding=2 to handle non-divisible sizes

    def forward(self, x):
        """
        Args:
            x: (batch, channels, height, width) image tensor
        Returns:
            (batch, n_patches, d) patch embeddings
        """
        # YOUR CODE HERE
        # 1. Apply the convolution
        # 2. Reshape from (batch, d, h', w') to (batch, h'*w', d)
        pass

Test it:

In [ ]:
patch = Patch(patch_size=4, in_channels=1, d=32)
x = t.randn(2, 1, 28, 28)
tokens = patch(x)
assert tokens.shape == (2, 49, 32), f"Expected (2, 49, 32), got {tokens.shape}"
print(f"Image {x.shape} -> Tokens {tokens.shape}")

---

### Exercise 2 — Implement UnPatch (Tokens → Image)

Difficulty: 🔴🔴🔴⭕⭕  |  Importance: 🔵🔵🔵🔵🔵  |  ~15 min

The inverse operation: convert a sequence of token embeddings back into an image. Each token is linearly projected to `patch_size^2 * out_channels` values, then reshaped back to spatial form.

In [ ]:
class UnPatch(nn.Module):
    """Convert patch token sequences back to images."""
    def __init__(self, patch_size=4, d=32, out_channels=1):
        super().__init__()
        self.out_channels = out_channels
        self.patch_size = patch_size
        # YOUR CODE HERE
        # Linear layer: d -> patch_size^2 * out_channels

    def forward(self, x):
        """
        Args:
            x: (batch, n_patches, d) patch embeddings
        Returns:
            (batch, out_channels, height, width) reconstructed image
        """
        # YOUR CODE HERE
        # 1. Linear projection to get pixel values per patch
        # 2. Reshape from (b, n_patches, ps*ps*c) to spatial form
        #    - first reshape to (b, h_patches, w_patches, c, ps, ps)
        #    - then permute and reshape to (b, c, H, W)
        pass

Test the round-trip (note: this won't reconstruct the image perfectly since Patch uses a learned conv, but the shapes should be correct):

In [ ]:
unpatch = UnPatch(patch_size=4, d=32, out_channels=1)
reconstructed = unpatch(tokens)
assert reconstructed.shape == (2, 1, 28, 28), f"Expected (2, 1, 28, 28), got {reconstructed.shape}"
print(f"Tokens {tokens.shape} -> Image {reconstructed.shape}")

## Part B: Transformer Building Blocks

Now we build the transformer blocks that will process our patch sequences. The key difference from a standard transformer is **adaptive layer normalization (AdaLN)**: instead of fixed normalization, we modulate the normalized activations using the conditioning signal (timestep + class label). This is how the model knows *what* to generate and *how much noise* is in the input.

---

### Exercise 3 — Implement RMSNorm

Difficulty: 🔴⭕⭕⭕⭕  |  Importance: 🔵🔵🔵⭕⭕  |  ~5 min

Root Mean Square normalization—like LayerNorm but without centering. Simpler, slightly faster, and used in most modern architectures.

$$\text{RMSNorm}(x) = \frac{x}{\text{RMS}(x) + \epsilon} \cdot \gamma, \quad \text{RMS}(x) = \sqrt{\frac{1}{d}\sum_{i=1}^d x_i^2}$$

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, d=32, eps=1e-6):
        super().__init__()
        # YOUR CODE HERE

    def forward(self, x):
        # YOUR CODE HERE
        pass

---

### Exercise 4 — Implement Attention with QK-Norm

Difficulty: 🔴🔴🔴⭕⭕  |  Importance: 🔵🔵🔵🔵🔵  |  ~20 min

Implement multi-head attention. A small but important detail: modern DiT architectures apply RMSNorm to the queries and keys *separately* before computing the dot product. This stabilizes training.

Note: unlike in a language model, we do **not** use a causal mask here. Each patch should attend to all other patches (bidirectional attention). We're denoising the whole image at once, not generating it left-to-right.

In [ ]:
class Attention(nn.Module):
    def __init__(self, d=32, n_head=4):
        super().__init__()
        self.n_head = n_head
        self.d = d
        self.d_head = d // n_head
        # YOUR CODE HERE
        # - QKV projection (single linear, 3*d output)
        # - Output projection
        # - RMSNorm for queries and keys (applied per-head, so dim = d_head)

    def forward(self, x):
        """
        Args:
            x: (batch, seq, d)
        Returns:
            (batch, seq, d)
        """
        # YOUR CODE HERE
        # 1. Project to Q, K, V
        # 2. Reshape to (batch, seq, n_head, d_head)
        # 3. Apply QK-norm
        # 4. Compute attention (no causal mask!)
        # 5. Apply output projection
        pass

---

### Exercise 5 — Implement Gated MLP

Difficulty: 🔴🔴⭕⭕⭕  |  Importance: 🔵🔵🔵⭕⭕  |  ~10 min

The feedforward network uses a gated architecture: `output = down(up(x) * SiLU(gate(x)))`. The gating mechanism allows the network to selectively pass information.

In [ ]:
class MLP(nn.Module):
    def __init__(self, d=32, exp=2):
        super().__init__()
        # YOUR CODE HERE

    def forward(self, x):
        # YOUR CODE HERE
        pass

---

### Exercise 6 — Implement Sinusoidal Embeddings for Timesteps

Difficulty: 🔴🔴⭕⭕⭕  |  Importance: 🔵🔵🔵🔵⭕  |  ~10 min

We need to encode the noise level (timestep) as a vector. We use sinusoidal embeddings, similar to positional encodings in the original transformer. These provide a smooth, continuous embedding for integer indices.

For index \(n\) and dimension \(i\):

$$\text{emb}(n, 2i) = \sin(n \cdot C^{-i/(d/2)}), \quad \text{emb}(n, 2i+1) = \cos(n \cdot C^{-i/(d/2)})$$

where \(C\) is a base frequency (typically 500 or 10000).

In [ ]:
class NumEmbedding(nn.Module):
    """Sinusoidal embeddings for integer values (used for timesteps)."""
    def __init__(self, n_max, d=32, C=500):
        super().__init__()
        # YOUR CODE HERE
        # Precompute the embedding table of shape (n_max, d) and register as buffer

    def forward(self, x):
        """
        Args:
            x: integer tensor of any shape
        Returns:
            embedding tensor with last dim = d
        """
        # YOUR CODE HERE
        pass

Test it:

In [ ]:
emb = NumEmbedding(1000, d=32)
out = emb(t.tensor([0, 100, 500, 999]))
assert out.shape == (4, 32)
# Check that different timesteps produce different embeddings
assert not t.allclose(out[0], out[1])
print("NumEmbedding works!")

---

### Exercise 7 — Implement the DiT Block with AdaLN Modulation

Difficulty: 🔴🔴🔴🔴⭕  |  Importance: 🔵🔵🔵🔵🔵  |  ~25 min

This is the key architectural innovation. Instead of standard `norm → attention → residual`, the DiT block uses **adaptive instance normalization**: the conditioning signal (timestep + class) produces 6 modulation parameters per block — scale, bias, and gate for both the attention and MLP sub-layers.

The forward pass for each sub-layer is:

In [ ]:
x_norm = RMSNorm(x)
x_modulated = x_norm * (1 + scale) + bias     # AdaLN
x_out = sublayer(x_modulated) * gate           # gating
x = x + x_out                                  # residual

The conditioning vector \(c\) is projected to produce all 6 parameters: `(scale1, bias1, gate1, scale2, bias2, gate2) = Linear(c).chunk(6)`

In [ ]:
class DiTBlock(nn.Module):
    def __init__(self, d=32, n_head=4, exp=2):
        super().__init__()
        # YOUR CODE HERE
        # - Two RMSNorm layers (one for attn, one for MLP)
        # - Attention and MLP modules
        # - Linear layer mapping d -> 6*d for modulation parameters

    def forward(self, x, c):
        """
        Args:
            x: (batch, seq, d) token embeddings
            c: (batch, d) conditioning vector (timestep + class embedding)
        Returns:
            (batch, seq, d) output
        """
        # YOUR CODE HERE
        # 1. Get 6 modulation params from c: scale1, bias1, gate1, scale2, bias2, gate2
        # 2. Apply modulated attention: norm -> scale & bias -> attn -> gate -> residual
        # 3. Apply modulated MLP: norm -> scale & bias -> mlp -> gate -> residual
        # Note: c is (batch, d) but x is (batch, seq, d), so unsqueeze c for broadcasting
        pass

---

### Exercise 8 — Assemble the Full DiT Model

Difficulty: 🔴🔴🔴⭕⭕  |  Importance: 🔵🔵🔵🔵🔵  |  ~20 min

Put all the pieces together into a complete DiT model. The model:


- Patchifies the input image into tokens

- Adds learnable positional embeddings

- Combines timestep embedding + class embedding into a conditioning vector

- Runs through \(N\) DiT blocks

- Applies final modulated normalization

- Unpatchifies back to image space

The model predicts the **velocity field** \(v\) — we'll explain what this means in the next section.

In [ ]:
class DiT(nn.Module):
    def __init__(self, h=28, w=28, n_classes=10, in_channels=1,
                 patch_size=4, n_blocks=4, d=32, n_head=4, exp=2, T=1000):
        super().__init__()
        self.T = T
        # YOUR CODE HERE
        # - Patch module
        # - Learnable positional embedding: (1, n_patches, d)
        # - NumEmbedding for timesteps (n_max=T)
        # - nn.Embedding for class labels (n_classes entries)
        # - SiLU activation for combining embeddings
        # - n_blocks DiTBlocks
        # - Final RMSNorm + modulation (Linear d -> 2*d for final scale, bias)
        # - UnPatch module

    def forward(self, x, c, ts):
        """
        Args:
            x: (batch, channels, H, W) noisy image
            c: (batch,) integer class labels
            ts: (batch,) float timesteps in [0, 1]
        Returns:
            (batch, channels, H, W) predicted velocity
        """
        # YOUR CODE HERE
        # 1. Convert float timestep to int: ts_int = min(ts * T, T-1)
        # 2. Combine embeddings: cond = SiLU(timestep_emb + class_emb)
        # 3. Patchify input and add positional embedding
        # 4. Run through DiT blocks
        # 5. Final modulated norm
        # 6. Unpatchify and return
        pass

    @property
    def device(self):
        return next(self.parameters()).device

    @property
    def dtype(self):
        return next(self.parameters()).dtype

Test it:

In [ ]:
model = DiT(h=28, w=28, n_classes=11, d=32, n_head=4, n_blocks=4).to(device)
x = t.randn(2, 1, 28, 28, device=device)
c = t.tensor([3, 7], device=device)
ts = t.tensor([0.5, 0.8], device=device)
v = model(x, c, ts)
assert v.shape == (2, 1, 28, 28), f"Expected (2, 1, 28, 28), got {v.shape}"
n_params = sum(p.numel() for p in model.parameters())
print(f"DiT output shape: {v.shape}, Parameters: {n_params:,}")

## Part C: Rectified Flow Matching

Now we get to the generative modeling objective. **Rectified flow matching** is elegant in its simplicity.

The idea: we define a straight-line path from each data point \(x\) to a random noise sample \(z \sim \mathcal{N}(0, I)\):

$$x_t = (1-t) \cdot x + t \cdot z$$

where \(t \in [0, 1]\). At \(t=0\), we have the clean data; at \(t=1\), we have pure noise.

The **velocity** along this path is:

$$v = \frac{dx_t}{dt} = z - x$$

We train the model to predict this velocity given \(x_t\) and \(t\). At inference time, we start from noise (\(t=1\)) and integrate backwards to \(t=0\) using Euler steps.

---

### Exercise 9 — Implement the Flow Matching Training Step

Difficulty: 🔴🔴🔴⭕⭕  |  Importance: 🔵🔵🔵🔵🔵  |  ~15 min

Implement a single training step. Given a batch of clean images and their labels:


- Sample random timesteps \(t \sim U(0, 1)\)

- Sample random noise \(z \sim \mathcal{N}(0, I)\)

- Compute the noisy image: \(x_t = (1 - t) \cdot x + t \cdot z\). Equivalently: \(x_t = x + t \cdot (z - x) = x - t \cdot v_{\text{true}}\) with \(v_{\text{true}} = x - z\)

- Predict the velocity: \(\hat{v} = \text{model}(x_t, c, t)\)

- Compute MSE loss: \(\mathcal{L} = \| \hat{v} - v_{\text{true}} \|^2\)

> **Note:**
**Sign convention note:** In our implementation, following the reference code, we define \(v_{\text{true}} = x - z\) (pointing from noise to data). The noisy image is then \(x_t = x - t \cdot v_{\text{true}}\). During sampling, we'll integrate from \(t=1\) (noise) to \(t=0\) (data) by subtracting velocity steps.

In [ ]:
def flow_matching_loss(model, x, c):
    """Compute the rectified flow matching loss.

    Args:
        model: DiT model
        x: (batch, channels, H, W) clean images
        c: (batch,) class labels

    Returns:
        scalar loss
    """
    # YOUR CODE HERE
    pass

---

### Exercise 10 — Implement Euler Sampling

Difficulty: 🔴🔴🔴⭕⭕  |  Importance: 🔵🔵🔵🔵🔵  |  ~15 min

To generate an image, we start from pure noise and step from \(t=1\) to \(t=0\). At each step, the model predicts the velocity, and we take an Euler step:

$$z_{i+1} = z_i + (t_i - t_{i+1}) \cdot \hat{v}(z_i, t_i)$$

We also use the **SD3 scheduler** which remaps the timestep schedule for better sample quality: `ts = 3*ts / (2*ts + 1)`.

In [ ]:
@t.no_grad()
def sample(model, z, y, n_steps=30, cfg=0):
    """Generate images from noise using Euler integration.

    Args:
        model: trained DiT
        z: (batch, channels, H, W) initial noise
        y: (batch,) class labels
        n_steps: number of Euler steps
        cfg: classifier-free guidance strength (0 = no guidance)

    Returns:
        (batch, channels, H, W) generated images
    """
    # YOUR CODE HERE
    # 1. Create n_steps+1 evenly spaced timesteps from 1 to 0
    # 2. Apply SD3 schedule: ts = 3*ts / (2*ts + 1)
    # 3. For each step:
    #    a. Predict velocity at current timestep
    #    b. If cfg > 0: also predict unconditional velocity (with y=0)
    #       and blend: v = v_uncond + cfg * (v_cond - v_uncond)
    #    c. Euler step: z = z + (ts[i] - ts[i+1]) * v_pred
    # 4. Return z
    pass

## Part D: Training on MNIST

---

### Exercise 11 — Train the Unconditional Model

Difficulty: 🔴🔴🔴⭕⭕  |  Importance: 🔵🔵🔵🔵🔵  |  ~20 min (+ training time)

Load MNIST and train the model. For the unconditional version, we ignore class labels (pass zeros). With a small model (\(d{=}32\), 4 blocks), training should converge in ~10 epochs on a single GPU.

In [ ]:
# Data loading
dataset = load_dataset("ylecun/mnist", split="train")

def collate_fn(batch):
    images = t.stack([t.tensor(item['image']).float() for item in batch])
    labels = t.stack([t.tensor(item['label']) for item in batch])
    images = images.unsqueeze(1)  # Add channel dim: (batch, 1, 28, 28)
    images = images / 127.5 - 1    # Normalize to [-1, 1]
    return images, labels

train_loader = DataLoader(dataset, batch_size=64, shuffle=True,
                          collate_fn=collate_fn, num_workers=2)

Now write the training loop:

In [ ]:
model = DiT(h=28, w=28, n_classes=11, d=32, n_head=4, n_blocks=4).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, betas=(0.9, 0.95))

# YOUR CODE HERE
# Write the training loop:
# - 10 epochs
# - For each batch: compute flow_matching_loss, backprop, clip gradients (max_norm=10), step
# - Pass class label = 0 (unconditional) for all samples
# - Print loss every 100 steps
# - After each epoch, generate and display samples

## Part E: Class-Conditional Generation & Classifier-Free Guidance

Now we add class conditioning. The key trick for **classifier-free guidance (CFG)** is:


- **During training:** randomly drop the class label (replace with 0) some fraction of the time (e.g., 20%). This teaches the model to generate both conditionally and unconditionally.

- **During inference:** run the model twice—once with the class label, once without—and extrapolate:
  $$v_{\text{guided}} = v_{\text{uncond}} + w \cdot (v_{\text{cond}} - v_{\text{uncond}})$$
  where \(w > 1\) amplifies the class signal.

---

### Exercise 12 — Train Class-Conditional Model with CFG

Difficulty: 🔴🔴🔴⭕⭕  |  Importance: 🔵🔵🔵🔵🔵  |  ~15 min (+ training time)

Modify the training loop to use actual class labels (shifted by 1, so label 0 is reserved for "unconditional") and apply 20% random dropout of the class label.

In [ ]:
# YOUR CODE HERE
# Modify the training loop:
# 1. Use actual class labels: c = labels + 1  (shift so 0 = unconditional)
# 2. Apply 20% label dropout: randomly set some labels to 0
#    mask = t.rand(batch_size) < 0.2
#    c[mask] = 0
# 3. Train for 10 epochs
# 4. After training, sample with CFG:
#    - For each class 1..10, generate samples with cfg=3
#    - Display in a grid showing each digit class

> **Note:**
**What to expect:** After 10 epochs, you should see recognizable digits. With CFG strength of 3, the digits should be clear and well-formed. Try varying the CFG strength: `cfg=0` (unconditional, blurry), `cfg=1` (mild guidance), `cfg=3` (strong guidance, sharp), `cfg=10` (over-saturated).

## Summary

In this exercise you've built a complete flow-matching image generator from scratch:


- **Patchify/UnPatchify:** Converting images to token sequences for transformer processing

- **DiT Architecture:** Transformer blocks with AdaLN modulation for conditioning

- **Rectified Flow Matching:** Training objective that learns to predict velocity fields along straight paths between data and noise

- **Euler Sampling:** Generating images by integrating the learned velocity field from noise to data

- **Classifier-Free Guidance:** Amplifying class conditioning at inference time for sharper results

In [Exercise 3](), we'll extend this to video generation—turning the image model into a frame-autoregressive world model for Pong.

[← Previous: Exercise 1]()   [Next: Exercise 3 →]()